# RTFM Training on Google Colab

Train RTFM (Robust Temporal Feature Magnitude Learning) on ShanghaiTech dataset.

**Prerequisites:** Upload `rtfm_colab.zip` to your Google Drive root folder.

In [2]:
# Step 1: Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [5]:
# Step 2: Mount Google Drive and unzip data
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/rtfm/data'):
    !unzip -q /content/drive/MyDrive/rtfm_colab.zip -d /content/rtfm
    print('Unzipped successfully')
else:
    print('Already unzipped')

!ls /content/rtfm/
!echo '---'
!echo "Train features:" && ls /content/rtfm/data/SH_Train_ten_crop_i3d/ | wc -l
!echo "Test features:" && ls /content/rtfm/data/SH_Test_ten_crop_i3d/ | wc -l

In [6]:
# Step 3: Fix list file paths (replace local paths with Colab paths)
import re

for list_file in ['shanghai-i3d-train-10crop.list', 'shanghai-i3d-test-10crop.list']:
    path = f'/content/rtfm/list/{list_file}'
    with open(path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        fname = line.strip().split('/')[-1]
        if 'Train' in list_file or ('Train' in line):
            new_lines.append(f'/content/rtfm/data/SH_Train_ten_crop_i3d/{fname}\n')
        else:
            new_lines.append(f'/content/rtfm/data/SH_Test_ten_crop_i3d/{fname}\n')
    
    with open(path, 'w') as f:
        f.writelines(new_lines)
    print(f'{list_file}: {len(new_lines)} entries — paths updated')

# Verify
with open('/content/rtfm/list/shanghai-i3d-train-10crop.list') as f:
    print(f'First train path: {f.readline().strip()}')
with open('/content/rtfm/list/shanghai-i3d-test-10crop.list') as f:
    print(f'First test path: {f.readline().strip()}')

In [7]:
# Step 4: Train RTFM
%cd /content/rtfm
!python -u train_rtfm.py --max-epoch 15000 2>&1 | tee training_log.txt

In [ ]:
# Step 6: Verify saved checkpoint AUC on Colab (CUDA)
import torch, sys, numpy as np
sys.path.insert(0, '/content/rtfm')
from model import Model
from dataset import Dataset
import option
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, auc

args = option.parser.parse_args([])
device = torch.device('cuda')
print("hello")
model = Model(args.feature_size, args.batch_size)
ckpt = torch.load('/content/rtfm/ckpt/rtfm_best.pkl', map_location='cuda')
model.load_state_dict(ckpt)
model = model.to(device)
model.eval()

test_loader = DataLoader(Dataset(args, test_mode=True),
    batch_size=1, shuffle=False, num_workers=0)

pred = torch.zeros(0, device=device)
with torch.no_grad():
    for i, inp in enumerate(test_loader):
        inp = inp.to(device)
        inp = inp.permute(0, 2, 1, 3)
        _, _, _, _, _, _, logits, _, _, _ = model(inputs=inp)
        logits = torch.squeeze(logits, 1)
        logits = torch.mean(logits, 0)
        pred = torch.cat((pred, logits))

gt = np.load('/content/rtfm/list/gt-sh.npy')
pred_np = np.repeat(pred.cpu().numpy(), 16)

print(f'Pred length: {len(pred_np)}, GT length: {len(gt)}')

fpr, tpr, _ = roc_curve(gt, pred_np)
rec_auc = auc(fpr, tpr)
print(f'\nCheckpoint AUC on Colab (CUDA): {rec_auc:.4f}')
print(f'Pred stats: min={pred.min():.4f}, max={pred.max():.4f}, mean={pred.mean():.4f}')

In [8]:
# Step 5: Copy trained checkpoints back to Google Drive
import shutil
os.makedirs('/content/drive/MyDrive/rtfm_checkpoints', exist_ok=True)

for ckpt in ['ckpt/rtfm_best.pkl', 'ckpt/rtfm_final.pkl']:
    src = f'/content/rtfm/{ckpt}'
    if os.path.exists(src):
        dst = f'/content/drive/MyDrive/rtfm_checkpoints/{os.path.basename(ckpt)}'
        shutil.copy2(src, dst)
        print(f'Copied {ckpt} -> Google Drive ({os.path.getsize(src)/1e6:.1f} MB)')

# Also save the training log
if os.path.exists('/content/rtfm/training_log.txt'):
    shutil.copy2('/content/rtfm/training_log.txt', '/content/drive/MyDrive/rtfm_checkpoints/training_log.txt')
    print('Copied training_log.txt -> Google Drive')

# Copy fpr/tpr for ROC curve
for f in ['fpr.npy', 'tpr.npy', 'precision.npy', 'recall.npy']:
    src = f'/content/rtfm/{f}'
    if os.path.exists(src):
        shutil.copy2(src, f'/content/drive/MyDrive/rtfm_checkpoints/{f}')

print('\nAll checkpoints saved to Google Drive: MyDrive/rtfm_checkpoints/')